# v33: Pure E1 Replica (target: reproduce 0.68)

**Exact E1 config** from `kaggle_scripts/sft_old/sfttrainer-training.ipynb`:
- Data: train.csv, stratified 100/type = 600 samples, seed=42
- SUFFIX: Official METRIC_SUFFIX (same as E1)
- LR: 2e-4, dropout: 0.05, 1 epoch, seq_len=1024
- LoRA: rank=32, alpha=16, all-linear
- Loss: Standard full-text (dataset_text_field)
- NO GRPO (pure SFT only, unlike E1 which had GRPO after but saved SFT as fallback)

In [ ]:
!pip install -q --no-index --find-links /kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages datasets trl --ignore-installed

In [ ]:
import datasets, trl
print(f'datasets: {datasets.__version__} | trl: {trl.__version__}')

In [ ]:
import os, sys, stat, shutil, gc, zipfile
import polars as pl
import torch
import torch.nn.functional as F
import kagglehub
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

# === Kaggle / Triton Environment Fixes ===
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell'
dst = '/tmp/ptxas-blackwell'
if os.path.exists(src):
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    import triton.backends.nvidia as nv_backend
    src_bin = os.path.join(os.path.dirname(nv_backend.__file__), 'bin')
    dst_bin = '/tmp/triton_nvidia_bin'
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
    os.environ['TRITON_PTXAS_PATH'] = dst

# ================================================================
#   v33 = EXACT E1 REPLICA (pure SFT, no GRPO)
# ================================================================
SFT_SAMPLES_PER_TYPE = 100  # 6 * 100 = 600 (E1 exact)
LORA_RANK = 32
LORA_ALPHA = 16
LORA_DROPOUT = 0.05       # E1 exact
MAX_SEQ_LEN = 1024
NUM_EPOCHS = 1
GRAD_ACCUM = 4
LR = 2e-4                 # E1 exact

OUTPUT_DIR = '/kaggle/working/adapter'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Config: samples/type={SFT_SAMPLES_PER_TYPE}, lr={LR}, epochs={NUM_EPOCHS}, '
      f'rank={LORA_RANK}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}')

## Data Loading & Formatting (E1 exact replica)

In [ ]:
MODEL_PATH = kagglehub.model_download(
    'metric/nemotron-3-nano-30b-a3b-bf16/transformers/default')

# Load original competition data (same as E1)
train_df = pl.read_csv('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv')
print(f'Total training samples: {len(train_df)}')

# --- Type classification (E1 exact) ---
def classify_type(prompt_text):
    p = prompt_text.lower()
    if 'bit manipulation' in p or '8-bit binary' in p: return 'bit_ops'
    elif 'encrypt' in p or 'decrypt' in p: return 'cipher'
    elif 'gravitational' in p or 'falling distance' in p: return 'gravity'
    elif 'numeral system' in p: return 'numeral'
    elif 'transformation rules' in p: return 'symbol'
    elif 'unit conversion' in p or 'convert the following measurement' in p: return 'unit_conv'
    return 'unknown'

train_df = train_df.with_columns(
    pl.col('prompt').map_elements(classify_type, return_dtype=pl.Utf8).alias('qtype')
)

# --- Stratified sampling (E1 exact: 100/type, seed=42) ---
def stratified_sample(df, n_per_type, seed):
    dfs = []
    for qtype in df['qtype'].unique().to_list():
        subset = df.filter(pl.col('qtype') == qtype)
        n = min(n_per_type, len(subset))
        dfs.append(subset.sample(n=n, seed=seed))
    return pl.concat(dfs)

sft_df = stratified_sample(train_df, SFT_SAMPLES_PER_TYPE, seed=42)
print(f'SFT samples: {len(sft_df)}')
for row in sft_df['qtype'].value_counts().sort('qtype').iter_rows():
    print(f'  {row[0]}: {row[1]}')

hf_dataset = Dataset.from_pandas(sft_df.drop('qtype').to_pandas())

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Official METRIC_SUFFIX (same as E1 and evaluation)
METRIC_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

def build_sft_text(example):
    user_msg = example['prompt'] + METRIC_SUFFIX
    assistant_msg = f'\\boxed{{{example["answer"]}}}'
    messages = [
        {'role': 'user', 'content': user_msg},
        {'role': 'assistant', 'content': assistant_msg},
    ]
    for kwargs in [{'enable_thinking': True}, {}]:
        try:
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False, **kwargs
            )
            return {'text': text}
        except Exception:
            continue
    return {'text': f'<|im_start|>user\n{user_msg}<|im_end|>\n<|im_start|>assistant\n{assistant_msg}<|im_end|>'}

# Template verification
_sample = build_sft_text({'prompt': 'What is 2+2?', 'answer': '4'})
print('Template preview:')
print(_sample['text'])

hf_dataset = hf_dataset.map(
    build_sft_text,
    remove_columns=hf_dataset.column_names,
)
print(f'\nSFT dataset: {len(hf_dataset)} samples')

## Model Loading & LoRA Configuration

In [ ]:
from unittest.mock import MagicMock
_mock_modules = [
    'cutlass', 'cutlass.cute', 'cutlass.utils',
    'mamba_ssm.ops.cute', 'mamba_ssm.ops.cute.mamba3',
    'mamba_ssm.ops.cute.mamba3.mamba3_step_fn',
    'mamba_ssm.ops.tilelang', 'mamba_ssm.ops.tilelang.mamba3',
    'mamba_ssm.ops.tilelang.mamba3.mamba3_mimo',
]
for mod_name in _mock_modules:
    if mod_name not in sys.modules:
        sys.modules[mod_name] = MagicMock()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, device_map='auto', trust_remote_code=True, dtype=torch.bfloat16
)
print(f'Model loaded. Vocab size: {len(tokenizer)}')

for name, mod in sys.modules.items():
    if 'modeling_nemotron_h' in name:
        mod.is_fast_path_available = False
        print(f'Patched {name}: is_fast_path_available = False')

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

# LoRA (E1 exact: rank=32, alpha=16, dropout=0.05)
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules='all-linear',
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Training (E1 exact config)

In [ ]:
import triton.backends.nvidia.compiler as nv_compiler
os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = '/tmp/ptxas-blackwell'
nv_compiler.get_ptxas_version = lambda arch: '12.0'

sft_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    logging_steps=10,
    bf16=True,
    max_grad_norm=1.0,
    optim='adamw_torch',
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    save_strategy='no',
    report_to='none',
    dataset_text_field='text',
    max_length=MAX_SEQ_LEN,
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': True},
)

trainer = SFTTrainer(
    model=model,
    train_dataset=hf_dataset,
    processing_class=tokenizer,
    args=sft_args,
)

total_steps = len(hf_dataset) // GRAD_ACCUM * NUM_EPOCHS
print(f'SFT: {len(hf_dataset)} samples x {NUM_EPOCHS} epoch = ~{total_steps} steps')
print(f'LR: {LR} | Max seq: {MAX_SEQ_LEN} | Dropout: {LORA_DROPOUT}')
print('Starting training...')
trainer.train()

## Save & Package Submission

In [ ]:
trainer.model.save_pretrained(OUTPUT_DIR)
print(f'Adapter saved to {OUTPUT_DIR}:')
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f'  {f} ({size/1024:.1f} KB)')

zip_path = '/kaggle/working/submission.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(OUTPUT_DIR):
        fpath = os.path.join(OUTPUT_DIR, fname)
        zf.write(fpath, fname)

print(f'Created {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)')

with zipfile.ZipFile(zip_path, 'r') as zf:
    names = zf.namelist()
    print(f'Contents: {names}')
    assert 'adapter_config.json' in names
    assert 'adapter_model.safetensors' in names
    print('submission.zip is valid and ready!')